# Préparation des données pour la prédiction de l'Éco-Score

Ce notebook permet de nettoyer les jeux de données `produits_avec_ecoscore.csv` et `produits_sans_ecoscore.csv` obtenus à l'étape précédente.

In [ ]:
import pandas as pd
import numpy as np
import io

## 1. Chargement des données

In [ ]:
# 1. Chargement du dataset AVEC Éco-Score (base d'apprentissage)
df_train = pd.read_csv('produits_avec_ecoscore.csv', dtype={'code': str}, low_memory=False)

# 2. Chargement du dataset SANS Éco-Score (à compléter)
df_predict = pd.read_csv('produits_sans_ecoscore.csv', dtype={'code': str}, low_memory=False)

print(f"Taille du dataset d'entraînement (AVEC Éco-Score) : {df_train.shape}")
print(f"Taille du dataset à compléter (SANS Éco-Score) : {df_predict.shape}")

## 2. Nettoyage des colonnes textuelles

In [ ]:
colonnes_textuelles = ['categories', 'labels', 'packaging', 'origins', 'ingredients_text', 'ingredients_analysis_tags']
colonnes_nutriments = ['energy_100g', 'sugars_100g', 'saturated-fat_100g', 'salt_100g', 'fiber_100g', 'proteins_100g', 'fruits-vegetables-legumes_100g']
colonnes_cibles = ['environmental_score_grade']

def nettoyer_texte(df, cols):
    for col in cols:
        if col in df.columns:
            # Remplacer les NaN par des chaînes vides
            df[col] = df[col].fillna('')
            # Tout mettre en minuscules
            df[col] = df[col].astype(str).str.lower()
            # Supprimer les sauts de ligne ou espaces multiples inutiles
            df[col] = df[col].str.replace('\n', ' ')
    return df

def preparer_donnees(df_input, is_train=False):
    df = df_input.copy()
    
    # On s'assure d'avoir un nom de produit
    df = df.dropna(subset=['product_name'])
    
    # Nettoyage du texte
    df = nettoyer_texte(df, colonnes_textuelles)
    
    # Conversion des colonnes nutritionnelles en numérique si nécessaire
    for col in colonnes_nutriments:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
            
    if is_train:
        # Pour l'entraînement, on ne garde que les lignes avec un Éco-Score valide
        df = df.dropna(subset=colonnes_cibles)
        df = df[df['environmental_score_grade'].isin(['a', 'b', 'c', 'd', 'e'])]
        
    return df

df_train_clean = preparer_donnees(df_train, is_train=True)
df_predict_clean = preparer_donnees(df_predict, is_train=False)

print(f"Train propre : {df_train_clean.shape}")
print(f"Predict propre : {df_predict_clean.shape}")

## 3. Export en CSV

In [ ]:
df_train_clean.to_csv('traite_avec_ecoscore.csv', index=False, encoding='utf-8-sig')
df_predict_clean.to_csv('traite_sans_ecoscore.csv', index=False, encoding='utf-8-sig')

print(f"✅ 'traite_avec_ecoscore.csv' sauvegardé ({len(df_train_clean)} lignes)")
print(f"✅ 'traite_sans_ecoscore.csv' sauvegardé ({len(df_predict_clean)} lignes)")